<a href="https://colab.research.google.com/github/vidhi-coder75/Customer-churn-prediction/blob/main/Customer_churn_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Apni file ka naam yahan likhein (agar alag naam se save ki hai toh change kar lena)
file_name = 'Churn_dataset.csv'

# Data load kar rahe hain
df = pd.read_csv(file_name)

# Data ki shuruaat ki 5 rows dekhein
print("Data Successfully Loaded! 🎉\n")
df.head()

In [ ]:
import pandas as pd
import numpy as np

# Set random seed for reproducibility (taaki data bar-bar change na ho)
np.random.seed(42)
n_customers = 2000

# Generate basic features
customer_ids = ['CUST-' + str(i).zfill(4) for i in range(1, n_customers + 1)]
tenure = np.random.randint(1, 73, n_customers)
contract_types = np.random.choice(['Month-to-month', 'One year', 'Two year'], n_customers, p=[0.5, 0.25, 0.25])
tech_support = np.random.choice(['Yes', 'No'], n_customers, p=[0.3, 0.7])
monthly_charges = np.random.uniform(20.0, 120.0, n_customers)

# Create DataFrame
df = pd.DataFrame({
    'CustomerID': customer_ids,
    'Tenure_Months': tenure,
    'Contract_Type': contract_types,
    'Tech_Support': tech_support,
    'Monthly_Charges': monthly_charges
})

# Generate 'Churn' column based on logical business rules
def determine_churn(row):
    prob = 0.20  # Base churn probability

    # Rule 1: Month-to-month & High charges -> High probability of leaving
    if row['Contract_Type'] == 'Month-to-month' and row['Monthly_Charges'] > 70:
        prob += 0.40

    # Rule 2: Long tenure & Two year contract -> Low probability of leaving
    if row['Tenure_Months'] > 48 and row['Contract_Type'] == 'Two year':
        prob -= 0.15

    # Rule 3: No Tech Support -> Slightly higher probability of leaving
    if row['Tech_Support'] == 'No':
        prob += 0.10

    # Ensure probability is strictly between 0 and 1
    prob = max(0, min(1, prob))

    # Assign 'Yes' or 'No' based on the calculated probability
    return np.random.choice(['Yes', 'No'], p=[prob, 1 - prob])

# Apply the logic to create the Churn column
df['Churn'] = df.apply(determine_churn, axis=1)

print("Perfect Dataset Created Successfully! 🎉\n")
print(df.head())

In [ ]:
# 1. Visualize the Overall Churn Rate
plt.figure(figsize=(6, 6))
df['Churn'].value_counts().plot(kind='pie', autopct='%1.1f%%', colors=['#66b3ff','#ff9999'], startangle=90)
plt.title('Overall Customer Churn Distribution')
plt.ylabel('') # Hides the y-label for a cleaner look
plt.show()

In [ ]:
# 2. Visualize Churn based on Contract Type
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='Contract_Type', hue='Churn', palette='Set2')
plt.title('Churn Rate by Contract Type')
plt.xlabel('Contract Type')
plt.ylabel('Number of Customers')
plt.show()

In [ ]:
# 3. Visualize Churn vs. Numerical Features (Tenure & Charges)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot for Monthly Charges
sns.boxplot(data=df, x='Churn', y='Monthly_Charges', ax=axes[0], palette='coolwarm')
axes[0].set_title('Impact of Monthly Charges on Churn')

# Boxplot for Tenure
sns.boxplot(data=df, x='Churn', y='Tenure_Months', ax=axes[1], palette='coolwarm')
axes[1].set_title('Impact of Customer Tenure on Churn')

plt.tight_layout()
plt.show()

In [ ]:
# Import Machine Learning libraries
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# --- 1. Data Preprocessing ---
# Hum CustomerID hata rahe hain kyunki wo prediction (pattern) mein kaam nahi aayega
df_ml = df.drop('CustomerID', axis=1)

# Binary text columns ko 1 aur 0 mein convert kar rahe hain
df_ml['Churn'] = df_ml['Churn'].map({'Yes': 1, 'No': 0})
df_ml['Tech_Support'] = df_ml['Tech_Support'].map({'Yes': 1, 'No': 0})

# Contract_Type (jisme 3 categories hain) ke liye One-Hot Encoding ka use kar rahe hain
df_ml = pd.get_dummies(df_ml, columns=['Contract_Type'], drop_first=True)

# --- 2. Train-Test Split ---
# Features (X) aur Target (y) ko alag karna
X = df_ml.drop('Churn', axis=1)
y = df_ml['Churn']

# Data ko 80% Training aur 20% Testing mein baat rahe hain
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- 3. Initialize Models ---
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42)
}

# --- 4. Train and Evaluate Models ---
print("--- Model Evaluation Results ---\n")

for name, model in models.items():
    # Model ko training data par train karein
    model.fit(X_train, y_train)

    # Testing data par predictions karein
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] # ROC-AUC ke liye probabilities

    # Metrics calculate karein (Guidelines ke hisaab se)
    accuracy = accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)

    # Results print karein
    print(f"Model: {name}")
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print(f"Recall:   {recall * 100:.2f}%")
    print(f"ROC-AUC:  {roc_auc:.4f}\n")
    print("-" * 32)

In [ ]:
# Random Forest model se 'Feature Importance' nikalna
rf_model = models["Random Forest"]
importances = rf_model.feature_importances_
features = X.columns

# Dataframe banakar sorting karna
feat_imp_df = pd.DataFrame({'Feature': features, 'Importance': importances})
feat_imp_df = feat_imp_df.sort_values(by='Importance', ascending=False)

# Bar chart plot karna
plt.figure(figsize=(10, 6))
sns.barplot(data=feat_imp_df, x='Importance', y='Feature', palette='viridis')
plt.title('Feature Importance: Why are Customers Churning?', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score', fontsize=12)
plt.ylabel('Customer Features', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
import joblib

# Hum apne Random Forest model ko save kar rahe hain
joblib.dump(models["Random Forest"], 'churn_model.pkl')

print("Model successfully saved as 'churn_model.pkl' ✅")


In [ ]:
%%writefile app.py
import streamlit as st
import joblib
import pandas as pd

# Save kiya hua model load karein
model = joblib.load('churn_model.pkl')

# Website ka Title aur Description
st.title("Customer Churn Prediction App 📊")
st.write("Enter customer details below to predict if they will leave the service.")

# User se input lene ke liye form/sliders
st.sidebar.header("Customer Information")
tenure = st.sidebar.slider("Tenure (Months)", 1, 72, 12)
monthly_charges = st.sidebar.number_input("Monthly Charges ($)", 20.0, 120.0, 50.0)
contract_type = st.sidebar.selectbox("Contract Type", ["Month-to-month", "One year", "Two year"])
tech_support = st.sidebar.radio("Has Tech Support?", ["Yes", "No"])

# Predict Button
if st.button("Predict Churn"):

    # User ke text input ko numbers mein convert karna (jaise model ko chahiye)
    tech_sup_val = 1 if tech_support == "Yes" else 0
    contract_one = 1 if contract_type == "One year" else 0
    contract_two = 1 if contract_type == "Two year" else 0

    # Input data ko table format mein lana
    input_data = pd.DataFrame({
        'Tenure_Months': [tenure],
        'Tech_Support': [tech_sup_val],
        'Monthly_Charges': [monthly_charges],
        'Contract_Type_One year': [contract_one],
        'Contract_Type_Two year': [contract_two]
    })

    # Model se prediction karwana
    prediction = model.predict(input_data)[0]
    probability = model.predict_proba(input_data)[0][1]

    # Result screen par dikhana
    st.subheader("Prediction Result:")
    if prediction == 1:
        st.error(f"⚠️ High Risk! This customer is likely to CHURN. (Probability: {probability*100:.1f}%)")
        st.write("**Recommendation:** Offer a discount or upgrade them to a 1-year contract.")
    else:
        st.success(f"✅ Safe! This customer is likely to STAY. (Probability: {(1-probability)*100:.1f}%)")

In [ ]:
# Streamlit aur localtunnel install kar rahe hain
!pip install -q streamlit
!npm install localtunnel -g > /dev/null 2>&1

# Background mein website start kar rahe hain
!streamlit run app.py &>/dev/null &

# Password (IP) print kar rahe hain
import urllib
print("👉 STEP 1: Copy this Endpoint IP (Password):")
!wget -q -O - ipv4.icanhazip.com

print("\n👉 STEP 2: Click the link below and paste the password to view your Web App:")
# Localtunnel se link generate kar rahe hain
!npx localtunnel --port 8501

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd

# 1. User Inputs ke liye Dropdowns aur Sliders banayein
style = {'description_width': 'initial'}
tenure_widget = widgets.IntSlider(min=1, max=72, step=1, value=12, description='Tenure (Months):', style=style)
charges_widget = widgets.FloatSlider(min=20.0, max=120.0, step=0.5, value=50.0, description='Monthly Bill ($):', style=style)
contract_widget = widgets.Dropdown(options=['Month-to-month', 'One year', 'Two year'], value='Month-to-month', description='Contract Type:', style=style)
tech_widget = widgets.Dropdown(options=['Yes', 'No'], value='No', description='Tech Support?', style=style)

# Predict Button
button = widgets.Button(description="Predict Churn", button_style='success', tooltip='Click to predict')
output = widgets.Output()

# 2. Prediction ka logic
def on_button_clicked(b):
    with output:
        clear_output()
        # Text ko numbers mein convert karna
        tech_val = 1 if tech_widget.value == 'Yes' else 0
        cont_one = 1 if contract_widget.value == 'One year' else 0
        cont_two = 1 if contract_widget.value == 'Two year' else 0

        # User ka data table format mein set karna
        input_data = pd.DataFrame({
            'Tenure_Months': [tenure_widget.value],
            'Tech_Support': [tech_val],
            'Monthly_Charges': [charges_widget.value],
            'Contract_Type_One year': [cont_one],
            'Contract_Type_Two year': [cont_two]
        })

        # Make sure column order exactly matches training data
        input_data = input_data.reindex(columns=X.columns, fill_value=0)

        # Humara best model (Random Forest) load karke predict karna
        rf_model = models["Random Forest"]
        prediction = rf_model.predict(input_data)[0]
        prob = rf_model.predict_proba(input_data)[0][1]

        print("\n" + "="*40)
        if prediction == 1:
            print(f"⚠️ ALERT: High Risk of Churn!")
            print(f"Probability of leaving: {prob*100:.1f}%")
            print("Recommendation: Try offering a discount.")
        else:
            print(f"✅ SAFE: Customer is likely to stay.")
            print(f"Probability of leaving: {prob*100:.1f}%")
        print("="*40 + "\n")

# 3. Screen par UI dikhana
button.on_click(on_button_clicked)
display(widgets.VBox([tenure_widget, charges_widget, contract_widget, tech_widget, button, output]))

In [ ]:
import gradio as gr
import pandas as pd

# 1. Prediction ka logic (Backend)
def predict_churn(tenure, monthly_charges, contract_type, tech_support):
    # Text input ko numbers mein convert karna
    tech_val = 1 if tech_support == 'Yes' else 0
    cont_one = 1 if contract_type == 'One year' else 0
    cont_two = 1 if contract_type == 'Two year' else 0

    # Dataframe banana
    input_data = pd.DataFrame({
        'Tenure_Months': [tenure],
        'Tech_Support': [tech_val],
        'Monthly_Charges': [monthly_charges],
        'Contract_Type_One year': [cont_one],
        'Contract_Type_Two year': [cont_two]
    })

    # Columns ko training data ke hisaab se match karna
    input_data = input_data.reindex(columns=X.columns, fill_value=0)

    # Model se predict karwana
    rf_model = models["Random Forest"]
    prediction = rf_model.predict(input_data)[0]
    prob = rf_model.predict_proba(input_data)[0][1]

    # Result set karna
    if prediction == 1:
        return f"⚠️ HIGH RISK: The customer is likely to CHURN. (Probability: {prob*100:.1f}%)"
    else:
        return f"✅ SAFE: The customer will STAY. (Probability: {prob*100:.1f}%)"

# 2. Website ka UI (Frontend) banana
with gr.Blocks(theme=gr.themes.Soft()) as web_app:
    gr.Markdown("# 📊 Customer Churn Prediction Web App")
    gr.Markdown("Enter customer details below to predict if they will leave the telecom service.")

    with gr.Row():
        with gr.Column():
            tenure_input = gr.Slider(minimum=1, maximum=72, step=1, value=12, label="Tenure (Months)")
            charges_input = gr.Slider(minimum=20.0, maximum=120.0, step=0.5, value=50.0, label="Monthly Charges ($)")
            contract_input = gr.Dropdown(choices=['Month-to-month', 'One year', 'Two year'], value='Month-to-month', label="Contract Type")
            tech_input = gr.Radio(choices=['Yes', 'No'], value='No', label="Has Tech Support?")

            predict_btn = gr.Button("Predict Churn", variant="primary")

        with gr.Column():
            output_text = gr.Textbox(label="Prediction Result", lines=4)

    # Button click par function chalana
    predict_btn.click(
        fn=predict_churn,
        inputs=[tenure_input, charges_input, contract_input, tech_input],
        outputs=output_text
    )

# 3. Website ko internet par LIVE karna
web_app.launch(share=True)